# Bilayer hBN RPA analysis and presentation plots

This notebook collects the material's electronic-structure, momentum-resolved RPA polarization, equilibrium method comparison, and effective dielectric-response plots.

In [ ]:
from pathlib import Path
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("default")
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["DejaVu Serif"],
    "mathtext.fontset": "stix",
    "figure.figsize": (7.1, 4.35),
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "semibold",
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.frameon": False,
    "legend.fontsize": 9,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.04,
    "pdf.fonttype": 42,
})

# Legacy settings retained below are overridden by the thesis style above.
plt.rcParams.update({
    "figure.figsize": (8.8, 5.0), "figure.dpi": 120,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.labelsize": 12, "axes.titlesize": 14,
    "legend.frameon": False, "legend.fontsize": 9, "savefig.dpi": 220,
})

def find_w90_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for parent in (start, *start.parents):
        if (parent / "data_analysis").is_dir() and (parent / "mos2").is_dir():
            return parent
    raise RuntimeError("Could not find w90 root.")

def load(path, *, mmap=False):
    return np.load(path, mmap_mode="r" if mmap else None, allow_pickle=True)

def nearest_index(values, target):
    return int(np.argmin(np.abs(values - target)))

def export_figure(fig, filename):
    path = PRESENTATION_OUTPUTS / filename
    fig.savefig(path, bbox_inches="tight")
    print("Exported:", path)
    return path

W90_ROOT = find_w90_root()
DATA_ANALYSIS_ROOT = W90_ROOT / "data_analysis"
VALIDATION_ROOT = DATA_ANALYSIS_ROOT / "validation_outputs/cnt_hbn_general/hbn"
PRESENTATION_OUTPUTS = DATA_ANALYSIS_ROOT / "presentation_outputs" / "hbn"
PRESENTATION_OUTPUTS.mkdir(parents=True, exist_ok=True)
print("Presentation outputs:", PRESENTATION_OUTPUTS)

## 1. Electronic-structure reference

In [ ]:
BAND_DATA = VALIDATION_ROOT / "band_structure" / "validation_hbn_bandstructure_GKM_shifted_Ef.npz"
band = load(BAND_DATA)
k_path = band["k_points"]
band_energies = band["plotted_eigenvalues"]
ticks = band["tick_positions"]
labels = band["tick_labels"]
fig, ax = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
for values in band_energies.T:
    ax.plot(k_path, values, color="#1f4e79", linewidth=0.9)
ax.axhline(0.0, color="#b22222", linestyle="--", linewidth=1.1)
ax.set_xticks(ticks)
ax.set_xticklabels([r"$\Gamma$" if label == "G" else label for label in labels])
for tick in ticks: ax.axvline(tick, color="#666666", alpha=0.25, linewidth=0.7)
ax.set(xlabel="Wave-vector path", ylabel=r"$E-E_F$ (eV)", title="Bilayer hBN Band Structure")
ax.set_xlim(float(k_path[0]), float(k_path[-1])); ax.set_ylim(-8, 8)
ax.grid(True, alpha=0.25, linewidth=0.5)
export_figure(fig, "hbn_bilayer_bandstructure.png")
plt.show()

## 2. Momentum-resolved equilibrium RPA polarization

Each curve is evaluated at one momentum transfer q; these are not q-averaged traces.

In [ ]:
POLARIZATION_DATA = VALIDATION_ROOT / "polarization_behavior" / "validation_hbn_rpa_polarization.npz"
pol = load(POLARIZATION_DATA)
q_points = pol["q_points"]; frequencies_eV = pol["frequencies"]; p_base = pol["p_base"]
q_fractions_to_plot = [0.25, 0.50]
for q_fraction in q_fractions_to_plot:
    q_index = nearest_index(q_points, q_fraction * np.pi)
    q_label = r"$q = \pi/4$" if np.isclose(q_fraction, 0.25, atol=0.02) else r"$q = \pi/2$"
    q_filename = "q_pi_over_4" if np.isclose(q_fraction, 0.25, atol=0.02) else "q_pi_over_2"
    fig, ax = plt.subplots(figsize=(10.8, 5.0), constrained_layout=True)
    ax.plot(
        frequencies_eV,
        p_base[q_index].real,
        color="tab:blue",
        linewidth=1.8,
        label=r"Re $P(q,\omega)$",
    )
    ax.plot(
        frequencies_eV,
        p_base[q_index].imag,
        color="tab:orange",
        linewidth=1.8,
        label=r"Im $P(q,\omega)$",
    )
    ax.axhline(0.0, color="0.35", linewidth=0.7)
    ax.set(
        title=f"Bilayer hBN RPA Polarization at {q_label}",
        xlabel="Energy transfer, $\\hbar\\omega$ (eV)",
        ylabel="Polarization response",
    )
    ax.legend()
    ax.grid(True, alpha=0.25, linewidth=0.5)
    export_figure(fig, f"hbn_rpa_polarization_{q_filename}.png")
    plt.show()

## 3. Transition energies for fixed q

The next figure compares the allowed band-to-band transition energy
$(\Delta E_{nm}(k,q)=E_{m,k+q}-E_{n,k})$ for the same fixed q values
used in the RPA polarization plots, and the corresponding imaginary
part of $P(q,\omega)$. Vertical lines mark:
- **Red line** (min ΔE): the strict arithmetic minimum of all allowed transitions across the full k-grid.
- **Orange dashed line** (Im P edge): where the RPA response first becomes non-zero.

**Why they don't align:** The RPA response Im P is typically offset from the minimum ΔE because:
1. **Finite broadening:** The δ(ω−ΔE) is replaced by a Lorentzian with finite lifetime (η), smoothing the onset.
2. **Matrix-element weighting:** Transitions are weighted by band overlap / oscillator strength; the minimum ΔE may have negligible form-factor.
3. **JDOS convolution:** The RPA sum reflects the (weighted) joint density-of-states convolved with the broadening kernel, not the arithmetic minimum.

In [ ]:
BAND_DATA = VALIDATION_ROOT / "band_structure" / "validation_hbn_bandstructure_GKM_shifted_Ef.npz"
band = load(BAND_DATA)
k_path = band["k_points"]
eigenvalues = band["eigenvalues"]
tick_positions = band["tick_positions"]

# Use the full periodic k-grid and fold k+q into the Brillouin zone so
# the transition-energy calculation matches the periodic RPA mesh.
k0 = float(k_path[0])
kN = float(k_path[-1])
k_step = float(np.mean(np.diff(k_path)))
bz_width = (kN - k0) + k_step

def fold_into_bz(k_val):
    return ((k_val - k0) % bz_width) + k0

POLARIZATION_DATA = VALIDATION_ROOT / "polarization_behavior" / "validation_hbn_rpa_polarization.npz"
pol = load(POLARIZATION_DATA)
q_points = pol["q_points"]
frequencies_eV = pol["frequencies"]
p_base = pol["p_base"]

for q_fraction in (0.25, 0.50):
    q_value = q_fraction * np.pi
    q_label = r"$q = \pi/4$" if np.isclose(q_fraction, 0.25) else r"$q = \pi/2$"
    q_filename = "q_pi_over_4" if np.isclose(q_fraction, 0.25) else "q_pi_over_2"

    transition_ks = []
    transition_energies = []
    min_transition = np.full(k_path.size, np.nan)

    for k_index, k_value in enumerate(k_path):
        target_k = fold_into_bz(k_value + q_value)
        kq_index = nearest_index(k_path, target_k)

        occupied = np.where(eigenvalues[k_index] < 0.0)[0]
        unoccupied = np.where(eigenvalues[kq_index] > 0.0)[0]
        if occupied.size == 0 or unoccupied.size == 0:
            continue

        delta = eigenvalues[kq_index][unoccupied][:, None] - eigenvalues[k_index][occupied][None, :]
        positive = delta[delta > 0.0]
        if positive.size == 0:
            continue

        transition_ks.append(np.full(positive.size, k_value))
        transition_energies.append(positive)
        min_transition[k_index] = positive.min()

    if transition_ks:
        transition_ks = np.concatenate(transition_ks)
        transition_energies = np.concatenate(transition_energies)
    else:
        transition_ks = np.empty(0, dtype=float)
        transition_energies = np.empty(0, dtype=float)

    q_index = nearest_index(q_points, q_value)
    im_p = np.imag(p_base[q_index])
    
    # Onset detection: find where Im P first becomes non-zero (leftmost edge at 0.5% threshold)
    abs_im_p = np.abs(im_p)
    threshold_edge = 0.005 * np.max(abs_im_p)
    im_onset_idx = int(np.argmax(abs_im_p > threshold_edge))
    im_onset_energy = frequencies_eV[im_onset_idx]

    valid_mask = ~np.isnan(min_transition)
    if not np.any(valid_mask):
        raise RuntimeError(f"No valid transitions found for q={q_fraction}")
    onset_energy = float(np.nanmin(min_transition))

    fig, (ax_transition, ax_im) = plt.subplots(1, 2, figsize=(14.0, 5.0), constrained_layout=True)
    if transition_ks.size > 0:
        ax_transition.scatter(transition_ks, transition_energies, s=4, alpha=0.15, color="tab:purple", label="Allowed transitions")
    ax_transition.plot(k_path[valid_mask], min_transition[valid_mask], color="tab:red", linewidth=2.0, label="Minimum transition energy")
    ax_transition.set_title(f"Transition energies (full k-grid) for {q_label}")
    ax_transition.set_xlabel(r"$k$ (periodic mesh)")
    ax_transition.set_ylabel(r"Transition energy $\Delta E_{nm}(k,q)$ (eV)")
    ax_transition.legend(loc="upper left")
    ax_transition.grid(True, alpha=0.25, linewidth=0.5)

    ax_im.plot(frequencies_eV, im_p, color="tab:blue", linewidth=1.9, label=r"Im $P(q,\omega)$")
    ax_im.axvline(onset_energy, color="tab:red", linestyle="-", linewidth=1.7, label=f"Min ΔE onset {onset_energy:.2f} eV")
    ax_im.axvline(im_onset_energy, color="tab:orange", linestyle="--", linewidth=1.7, label=f"Im P edge {im_onset_energy:.2f} eV")
    ax_im.set_title(f"Imaginary polarization response, {q_label}")
    ax_im.set_xlabel("Energy transfer (eV)")
    ax_im.set_ylabel(r"Im $P(q,\omega)$")
    ax_im.legend(loc="upper right")
    ax_im.grid(True, alpha=0.25, linewidth=0.5)
    
    # Explain the energy shift in a text box
    shift_eV = im_onset_energy - onset_energy
    textstr = f"Note: Im P edge is {shift_eV:.2f} eV\nhigher than min ΔE due to\nfinite broadening & matrix\nelement weighting in RPA."
    ax_im.text(0.98, 0.05, textstr, transform=ax_im.transAxes, fontsize=9,
               verticalalignment='bottom', horizontalalignment='right',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    fig.suptitle(f"Transition energies and Im P(q,\omega) for {q_label}", fontsize=15)
    export_figure(fig, f"hbn_transition_energy_vs_imP_{q_filename}.png")
    plt.show()


## 3. Equilibrium polarization: finite-device GW/NEGF versus periodic RPA

The finite-device trace is normalized by the inferred device-to-unit-cell basis ratio. The periodic RPA curve is the q-average of unit-cell matrix traces.

In [ ]:
HBN_COMPARISON_ROOT = DATA_ANALYSIS_ROOT / "generated_equilibrium_comparisons/hbn"
RPA_RAW = HBN_COMPARISON_ROOT / "outputs_RPA_export/raw_rpa_debug"
NEGF_DENSITY = HBN_COMPARISON_ROOT / "outputs_NEGF_equilibrium/p_retarded_density_0.npy"
required = [RPA_RAW / "polarization_retarded_qw.npy", RPA_RAW / "frequencies_eV.npy", NEGF_DENSITY]
missing = [p for p in required if not p.exists()]
if missing:
    print("Skipping optional finite GW/NEGF comparison; missing outputs:")
    for path in missing:
        print(" -", path)
else:
    p_rpa = load(RPA_RAW / "polarization_retarded_qw.npy", mmap=True)
    freq = np.asarray(load(RPA_RAW / "frequencies_eV.npy"))
    p_negf_density = np.asarray(load(NEGF_DENSITY, mmap=True))

    # Quatrex writes polarization_density as -diag(P^R)/(2*pi). Reconstruct the
    # raw finite-device trace, then normalize its nine repeated cells to one cell.
    rpa = np.asarray(np.trace(p_rpa, axis1=2, axis2=3)).mean(axis=0)
    num_orbitals_per_cell = p_rpa.shape[-1]
    cell_ratio = p_negf_density.shape[1] / num_orbitals_per_cell
    negf = (-2.0 * np.pi * p_negf_density.sum(axis=1)) / cell_ratio

    # Reject unstable OBC output instead of silently exporting a misleading plot.
    def spike_ratio(values):
        scale = np.percentile(np.abs(values), 95)
        return np.max(np.abs(np.diff(values))) / max(scale, 1e-15)

    quality = max(spike_ratio(negf.real), spike_ratio(negf.imag))
    if quality > 5.0:
        raise RuntimeError(
            "The dedicated hBN GW/NEGF polarization is dominated by isolated numerical "
            f"poles (spike ratio={quality:.1f}). Rerun the stabilized equilibrium config "
            "before using this comparison plot."
        )

    n = min(len(freq), len(rpa), len(negf))
    x = freq[:n]
    rpa = rpa[:n]
    negf = negf[:n]
    print("Finite-device/unit-cell basis ratio:", cell_ratio)
    fig, ax = plt.subplots(figsize=(8.8, 5.2), constrained_layout=True)
    ax.plot(x, negf.real, color="black", linewidth=1.7, label=f"Re finite GW/NEGF / {cell_ratio:g} cells")
    ax.plot(x, negf.imag, color="black", linestyle="--", linewidth=1.7, label=f"Im finite GW/NEGF / {cell_ratio:g} cells")
    ax.plot(x, rpa.real, color="tab:blue", linewidth=1.9, label="Re periodic q-avg RPA")
    ax.plot(x, rpa.imag, color="tab:blue", linestyle="--", linewidth=1.9, label="Im periodic q-avg RPA")
    ax.axhline(0.0, color="0.35", linewidth=0.7)
    ax.set(
        title="Bilayer hBN Equilibrium Polarization: GW/NEGF and RPA",
        xlabel="Energy transfer, $\\hbar\\omega$ (eV)",
        ylabel="Polarization trace per unit cell",
    )
    ax.legend(ncol=2)
    ax.grid(True, alpha=0.25, linewidth=0.5)
    export_figure(fig, "hbn_equilibrium_polarization_negf_vs_rpa.png")
    plt.show()


## 4. Absolute-magnitude validation: periodic RPA versus periodic Green-function bubble

This controlled validation uses the same periodic bilayer-hBN Bloch Hamiltonian,
occupations, momentum mesh, density vertex, broadening, and spin degeneracy in both
calculations. The independently integrated periodic Green-function bubble agrees with
the band-sum RPA imaginary response without applying a normalization factor.

This establishes the absolute spectral magnitude of the periodic RPA implementation.
The remaining magnitude difference in the finite-device comparison above therefore
belongs to the finite open-boundary representation and is not an RPA prefactor error.


In [ ]:
PERIODIC_VALIDATION = PRESENTATION_OUTPUTS / "hbn_periodic_rpa_vs_gf_bubble.npz"
if not PERIODIC_VALIDATION.exists():
    raise FileNotFoundError("Missing periodic RPA/GF validation data. Run data_analysis/scripts/validate_hbn_periodic_rpa_vs_gf_bubble.py first.")

periodic_validation = load(PERIODIC_VALIDATION)
validation_frequency = periodic_validation["frequencies_eV"]
validation_q_over_pi = periodic_validation["q_over_pi"]
validation_rpa = periodic_validation["rpa_trace"]
validation_gf = periodic_validation["gf_trace"]

for index, q_fraction in enumerate(validation_q_over_pi):
    q_label = r"$q \approx \pi/4$" if np.isclose(q_fraction, 0.25, atol=0.02) else r"$q = \pi/2$"
    q_filename = "q_pi_over_4" if np.isclose(q_fraction, 0.25, atol=0.02) else "q_pi_over_2"
    fig, axes = plt.subplots(1, 2, figsize=(12.8, 4.8), constrained_layout=True)

    for axis, component, component_label in zip(
        axes, ("real", "imag"), (r"$\mathrm{Re}\,P(q,\omega)$", r"$\mathrm{Im}\,P(q,\omega)$")
    ):
        rpa_component = getattr(validation_rpa[index], component)
        gf_component = getattr(validation_gf[index], component)
        component_error = np.linalg.norm(gf_component - rpa_component) / np.linalg.norm(rpa_component)

        axis.plot(validation_frequency, rpa_component, linewidth=2.1, label="Band-sum RPA")
        axis.plot(validation_frequency, gf_component, "--", linewidth=1.9, label="Periodic GF bubble")
        axis.axhline(0.0, color="0.35", linewidth=0.8)
        axis.set_title(f"{component_label}\nrelative error = {100 * component_error:.3f}%")
        axis.set_xlabel("Energy transfer (eV)")
        axis.set_ylabel("Polarization trace")
        axis.grid(True, alpha=0.25, linewidth=0.5)

    axes[0].legend()
    fig.suptitle(f"Bilayer hBN Periodic Polarization Validation, {q_label}")
    export_figure(fig, f"hbn_periodic_rpa_vs_gf_{q_filename}.png")
    plt.show()


## 5. Effective RPA dielectric response

The dielectric response and loss are shown for representative finite q-points using the saved effective projection.

In [ ]:
DIELECTRIC_DATA = VALIDATION_ROOT / "dielectric_function" / "validation_hbn_rpa_effective_dielectric.npz"
diel = load(DIELECTRIC_DATA)
q_points_d = diel["q_points"]; frequencies_d = diel["frequencies"]
epsilon_eff = diel["epsilon_eff"]; loss_eff = diel["loss_eff"]
projection = str(diel["projection"]) if "projection" in diel.files else "effective projection"
q_indices = [nearest_index(q_points_d, fraction * np.pi) for fraction in (0.25, 0.50)]
colors = ["tab:blue", "tab:orange"]
fig, axes = plt.subplots(1, 3, figsize=(13.8, 4.4), constrained_layout=True)
for q_index, color in zip(q_indices, colors):
    label = f"q/pi = {q_points_d[q_index] / np.pi:.2f}"
    axes[0].plot(frequencies_d, epsilon_eff[q_index].real, label=label, linewidth=1.9, color=color)
    axes[1].plot(frequencies_d, epsilon_eff[q_index].imag, label=label, linewidth=1.9, color=color)
    axes[2].plot(frequencies_d, loss_eff[q_index], label=label, linewidth=1.9, color=color)
axes[0].axhline(1.0, color="0.45", linewidth=0.8, linestyle=":")
axes[1].axhline(0.0, color="0.45", linewidth=0.8); axes[2].axhline(0.0, color="0.45", linewidth=0.8)
axes[0].set_title(r"Re $\epsilon_\mathrm{eff}(q,\omega)$")
axes[1].set_title(r"Im $\epsilon_\mathrm{eff}(q,\omega)$")
axes[2].set_title(r"Loss $-\mathrm{Im}[1/\epsilon_\mathrm{eff}]$")
for ax in axes: ax.set_xlabel("Energy transfer, $\\hbar\\omega$ (eV)"); ax.legend(); ax.grid(True, alpha=0.25, linewidth=0.5)
axes[0].set_ylabel("Effective dielectric response"); fig.suptitle("Bilayer hBN RPA Effective Dielectric Response", fontsize=15)
export_figure(fig, "hbn_rpa_effective_dielectric.png"); plt.show(); print("Projection:", projection)

## Presentation summary

Use the band structure and q-lines for RPA verification. The periodic
RPA-versus-periodic-GF comparison validates the RPA absolute spectral magnitude.
Use the finite-device GW/NEGF comparison only as a representation diagnostic; its
remaining magnitude difference is not a justified RPA normalization factor. The
effective dielectric response presents the resulting screening behavior. Exports are
written to `presentation_outputs/hbn/`.


## 6. Individual q-point dielectric response: real and imaginary components

Separate plots for each q-value showing the real and imaginary parts of the effective dielectric function. These highlight how screening evolves with momentum transfer.

In [ ]:
# Generate individual q-point plots for hBN
q_fractions_to_plot = [0.25, 0.50]  # q/pi = 0.25 and q/pi = 0.50

for q_fraction in q_fractions_to_plot:
    q_index = nearest_index(q_points_d, q_fraction * np.pi)
    q_actual = q_points_d[q_index] / np.pi
    
    # Create descriptive label
    if np.isclose(q_fraction, 0.25, atol=0.02):
        q_label = r"$q = \pi/4$"
        q_filename = "q_pi_over_4"
    elif np.isclose(q_fraction, 0.50, atol=0.02):
        q_label = r"$q = \pi/2$"
        q_filename = "q_pi_over_2"
    else:
        q_label = f"$q \\approx {q_actual:.2f}\\pi$"
        q_filename = f"q_{q_actual:.2f}pi"
    
    # Create figure with real and imaginary parts on the same axis
    fig, ax = plt.subplots(figsize=(10.8, 5.0), constrained_layout=True)
    ax.plot(
        frequencies_d,
        epsilon_eff[q_index].real,
        linewidth=2.2,
        color="tab:blue",
        label=r"Re $\epsilon_\mathrm{eff}(q,\omega)$",
    )
    ax.plot(
        frequencies_d,
        epsilon_eff[q_index].imag,
        linewidth=2.2,
        color="tab:orange",
        label=r"Im $\epsilon_\mathrm{eff}(q,\omega)$",
    )
    ax.axhline(1.0, color="0.45", linewidth=0.8, linestyle=":", label="Vacuum")
    ax.axhline(0.0, color="0.35", linewidth=0.7)
    ax.set_title(f"Bilayer hBN Effective Dielectric Response at {q_label}", fontsize=15)
    ax.set_xlabel("Energy transfer, $\\hbar\\omega$ (eV)")
    ax.set_ylabel("Effective dielectric response, $\\epsilon_\\mathrm{eff}$")
    ax.legend()
    ax.grid(True, alpha=0.25, linewidth=0.5)

    export_figure(fig, f"hbn_epsilon_eff_{q_filename}.png")
    plt.show()

## 7. Static 2D polarizability and effective dielectric constant

### Why the resonant peaks are not dielectric constants

The large values in the frequency-dependent dielectric plots occur at finite momentum and finite energy, where interband transitions resonantly enhance the response. A static material property must instead be extracted from the long-wavelength, zero-frequency limit, \(q\to0\) and \(\omega=0\).

### Thickness-independent 2D response

For a two-dimensional insulator, the static density response behaves as \(P(q,0)\propto-q^2\). The intrinsic static electronic 2D polarizability is therefore

\[
\alpha_{2D}
= -\lim_{q\to0}\frac{e^2 P(q,0)}{A_{\mathrm{cell}}q^2}.
\]

Here:

- \(P(q,0)\) is the static RPA polarization summed over the uniform density channel;
- \(q\) is the physical in-plane momentum transfer in \(\mathrm{\AA}^{-1}\);
- \(e^2=14.3996\ \mathrm{eV\,\AA}\);
- \(A_{\mathrm{cell}}=|\mathbf a_1\times\mathbf a_2|\) is the in-plane unit-cell area.

The supplied hBN structure has the **hexagonal**, rather than orthogonal, lattice vectors

\[
\mathbf a_1=(2.50399,0,0)\ \mathrm{\AA},\qquad
\mathbf a_2=(-1.251995,2.168519,0)\ \mathrm{\AA},
\]

which give

\[
A_{\mathrm{cell}}=5.42995\ \mathrm{\AA}^2.
\]

The calculation uses the full two-dimensional translation-block Hamiltonian, a \(480\times80\) k-grid, and ten small nonzero momenta approaching \(\Gamma\). A fit of

\[
\alpha_{2D}(q)=\alpha_{2D}(0)+Cq^2
\]

gives \(\alpha_{2D}(0)=3.493\ \mathrm{\AA}\). The variation over the sampled range is below 0.4%, demonstrating small-q convergence.

### Thickness-dependent dielectric constant

A freestanding 2D layer does not possess a unique dimensionless dielectric constant until an effective thickness \(d\) is assigned. In Gaussian units, the in-plane conversion is

\[
\epsilon_{\mathrm{eff},\parallel}(d)
=1+\frac{4\pi\alpha_{2D}}{d}.
\]

The value \(d=6.66\ \mathrm{\AA}\) is used as a **bulk-derived thickness convention**. Bulk hBN has a c-axis lattice parameter close to \(6.66\ \mathrm{\AA}\), containing two hBN layers; equivalently, each layer is assigned approximately one interlayer repeat of \(3.33\ \mathrm{\AA}\). Thus a bilayer is assigned \(2\times3.33=6.66\ \mathrm{\AA}\). This is not the literal distance between the two atomic planes, but a volume-per-layer convention used to convert a sheet response into a bulk-like relative permittivity.

Using this convention gives

\[
\epsilon_{\mathrm{eff},\parallel}(6.66\ \mathrm{\AA})\approx7.59.
\]

The simulation-cell height of \(26\ \mathrm{\AA}\) is not used as the material thickness because most of it is vacuum. Using it would instead produce the vacuum-diluted supercell value, approximately 2.69.

### Scope and literature comparison

This result contains the **electronic RPA contribution only** and is a head-only density response; ionic/phonon polarization and local-field coupling are excluded. Laturia, Van de Put, and Vandenberghe report that the in-plane static hBN permittivity changes only weakly from 6.82 for monolayer to 6.93 for bulk when both electronic and ionic contributions and their thickness convention are included. The present value of 7.59 is therefore comparable in scale, but it should not be described as an exact reproduction because the physical approximations and thickness definitions differ.

References:

1. A. Laturia, M. L. Van de Put, and W. G. Vandenberghe, *npj 2D Materials and Applications* **2**, 6 (2018), https://doi.org/10.1038/s41699-018-0050-x.
2. O. Adeniran and Z.-F. Liu, "Dielectric screening at TMD:hBN interfaces," arXiv:2212.01676 (2022), https://arxiv.org/abs/2212.01676.
3. A. Pierret *et al.*, "Dielectric permittivity, conductivity and breakdown field of hexagonal boron nitride," arXiv:2201.05826 (2022), https://arxiv.org/abs/2201.05826.

In [ ]:
STATIC_2D_DATA = (
    VALIDATION_ROOT
    / "dielectric_function"
    / "static_2d"
    / "hbn_2d_static_final.npz"
)
static_2d = load(STATIC_2D_DATA)
q_static = static_2d["q_magnitudes_inverse_angstrom"]
alpha_by_q = static_2d["alpha_2d_by_q_angstrom"]
alpha_2d = float(static_2d["alpha_2d_angstrom"])
fit_count = int(static_2d["fit_count"])
alpha_q2_slope = float(static_2d["alpha_q2_slope_angstrom3"])
fit_r_squared = float(static_2d["fit_r_squared"])
cell_area = float(static_2d["cell_area_angstrom2"])
variation_percent = 100.0 * (alpha_by_q.max() - alpha_by_q.min()) / alpha_2d

print(f"Bilayer hBN electronic alpha_2D = {alpha_2d:.5f} Angstrom")
print(f"Small-q extrapolation R^2 = {fit_r_squared:.8f}")
print(f"Smallest-q direct estimate = {alpha_by_q[0]:.5f} Angstrom")
print(f"In-plane cell area = {cell_area:.5f} Angstrom^2")
print(f"Variation across sampled q range = {variation_percent:.3f}%")
print("Effective epsilon depends on the selected thickness:")
for thickness in (6.66, 7.00, 7.50, 26.00):
    epsilon_thickness = 1.0 + 4.0 * np.pi * alpha_2d / thickness
    label = "simulation-cell height" if np.isclose(thickness, 26.0) else "assumed slab thickness"
    print(f"  d = {thickness:5.2f} Angstrom ({label}): epsilon_eff = {epsilon_thickness:.3f}")

fig, ax = plt.subplots(figsize=(8.8, 5.0), constrained_layout=True)
ax.plot(q_static, alpha_by_q, "o-", color="tab:blue", linewidth=2.0,
        label=r"$\alpha_{2D}(q)$")
q_fit = np.linspace(0.0, q_static[fit_count - 1], 150)
ax.plot(q_fit, alpha_2d + alpha_q2_slope * q_fit**2, "--",
        color="tab:orange", linewidth=1.8,
        label=rf"$q\to0$: $\alpha_{{2D}}={alpha_2d:.3f}$ Angstrom")
ax.scatter([0.0], [alpha_2d], marker="x", color="black", s=70)
ax.set_title("Bilayer hBN Static Electronic 2D Polarizability")
ax.set_xlabel(r"$|q|$ ($\mathrm{Angstrom}^{-1}$)")
ax.set_ylabel(r"$\alpha_{2D}$ (Angstrom)")
ax.grid(True, alpha=0.25, linewidth=0.5)
ax.legend()
plt.show()